# Check how far PPO reversed in each episode

In [ ]:
from pathlib import Path
import csv
import glob
import math
import os
import pandas as pd

BASE = str(Path.home() / 'greenhouse_ws')
episodes = pd.read_csv(f'{BASE}/runs/episodes_ppo.csv').set_index('world')

results = []
for file in glob.glob(f'{BASE}/runs/trajectories_ppo/*.csv'):
    world = os.path.basename(file)[:-4]
    points = []
    for row in csv.DictReader(open(file)):
        try:
            x, y, yaw = float(row['tx']), float(row['ty']), float(row['tyaw'])
        except ValueError:
            continue
        if math.isfinite(x) and math.isfinite(y) and math.isfinite(yaw):
            points.append((x, y, yaw))
    total, longest, current = 0.0, 0.0, 0.0
    for (x0, y0, _), (x1, y1, yaw) in zip(points, points[1:]):
        dx, dy = x1 - x0, y1 - y0
        step = math.hypot(dx, dy)
        if step > 0.002 and dx * math.cos(yaw) + dy * math.sin(yaw) < 0:
            current += step
        else:
            if current >= 0.10:
                total += current
                longest = max(longest, current)
            current = 0.0
    if current >= 0.10:
        total += current
        longest = max(longest, current)
    if total > 0:
        reached = episodes.loc[world, 'truth_dist_m'] < 0.75 if world in episodes.index else None
        results.append((world, round(total, 2), round(longest, 2), reached))

table = pd.DataFrame(results, columns=['world', 'reversed_m', 'longest_m', 'reached_goal'])
print(len(table), 'worlds where PPO reversed at least 10 cm in one go')
print(table.sort_values('reversed_m', ascending=False).head(15).to_string(index=False))

In [ ]:
print(len(glob.glob(f'{BASE}/runs/trajectories_ppo/*.csv')))